# Lens D — Methods Gaps

Are rigorous methods being applied evenly across global health topics, or are some areas relying on weak evidence? This notebook identifies systematic gaps in the topic × method matrix.

### Analyses
1. **Topic × method matrix** — observed vs. expected proportions with z-scores
2. **Top 15 under-represented topic-method cells** — where are the gaps?
3. **Geography × method** — causal inference usage by first-author country
4. **Methods diffusion curves** — when did each method spread across topics?
5. **Manual review of top 10 gap cells** — genuine opportunity vs. appropriate absence

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

DB = 'data/global_health.duckdb'
con = duckdb.connect(DB, read_only=True)

## Load core data

In [ ]:
# Taxonomy labels
topic_labels = pd.read_csv('data/taxonomy/topic_taxonomy.csv')
cat_names = topic_labels.drop_duplicates('category_letter')[['category_letter', 'category_name']]
cat_map = dict(zip(cat_names['category_letter'], cat_names['category_name']))

method_labels = pd.read_csv('data/taxonomy/methods_taxonomy.csv')
method_map = dict(zip(method_labels['method_id'], method_labels['method_name']))

# All classified works with topic and method
works = con.execute("""
    SELECT openalex_id, publication_year, topic_category, topic_subtopic,
           method_type, study_country
    FROM works
    WHERE topic_category IS NOT NULL AND method_type IS NOT NULL
""").fetchdf()

works['topic_name'] = works['topic_category'].map(cat_map)
works['method_name'] = works['method_type'].map(method_map)

print(f'Papers with both topic and method: {len(works):,}')
print(f'Topic categories: {works["topic_category"].nunique()}')
print(f'Method types: {works["method_type"].nunique()}')

---
## D1. Topic × Method Matrix — Observed vs. Expected

If topics and methods were independent, we'd expect a uniform distribution. The z-score measures how far the observed count deviates from what we'd expect under independence. Large negative z-scores indicate genuine gaps.

In [ ]:
# Observed cross-tabulation
observed = pd.crosstab(works['topic_name'], works['method_name'])
n_total = observed.values.sum()

# Expected under independence: E_ij = (row_total * col_total) / grand_total
row_totals = observed.sum(axis=1).values.reshape(-1, 1)
col_totals = observed.sum(axis=0).values.reshape(1, -1)
expected = (row_totals * col_totals) / n_total
expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)

# Z-score: (observed - expected) / sqrt(expected)
z_scores = (observed - expected_df) / np.sqrt(expected_df)

# Heatmap of z-scores
fig, ax = plt.subplots(figsize=(18, 12))
sns.heatmap(z_scores, annot=True, fmt='.1f', cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Z-score (red = over-represented, blue = under-represented)'})
ax.set_title('Topic × Method Z-Score Matrix\nPositive = over-represented, Negative = under-represented')
ax.set_ylabel('')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
plt.show()

---
## D2. Top 15 Under-Represented Topic-Method Cells

These are the largest gaps: topic-method combinations that appear far less often than expected. Each represents a potential research opportunity.

In [ ]:
# Flatten z-scores to find the most under-represented cells
gaps = []
for topic in z_scores.index:
    for method in z_scores.columns:
        gaps.append({
            'topic': topic,
            'method': method,
            'z_score': z_scores.loc[topic, method],
            'observed': observed.loc[topic, method],
            'expected': expected_df.loc[topic, method],
        })

gaps_df = pd.DataFrame(gaps)

# Filter to cells where we'd expect at least 3 papers (avoid tiny cells)
meaningful_gaps = gaps_df[gaps_df['expected'] >= 3].sort_values('z_score')

top_gaps = meaningful_gaps.head(15)

fig, ax = plt.subplots(figsize=(12, 8))
labels = [f"{row['topic']}\n× {row['method']}" for _, row in top_gaps.iterrows()]
ax.barh(range(len(top_gaps)), top_gaps['z_score'], color='#2171b5')
ax.set_yticks(range(len(top_gaps)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Z-score (more negative = larger gap)')
ax.set_title('Top 15 Under-Represented Topic × Method Combinations')
ax.invert_yaxis()

# Annotate with observed/expected
for i, (_, row) in enumerate(top_gaps.iterrows()):
    ax.text(row['z_score'] - 0.1, i,
            f"obs={row['observed']:.0f}, exp={row['expected']:.1f}",
            va='center', ha='right', fontsize=8, color='white')
fig.tight_layout()
plt.show()

print('\nTop 15 gaps (potential research opportunities):')
top_gaps[['topic', 'method', 'observed', 'expected', 'z_score']].to_string(index=False)

---
## D3. Geography × Method — Causal Inference Usage by First-Author Country

Are rigorous causal methods (RCTs, quasi-experiments) concentrated in certain countries? This reveals whether methodological capacity is distributed equitably.

In [ ]:
# Get first-author country for each paper
geo_methods = con.execute("""
    SELECT w.openalex_id, w.method_type,
           a.institution_country AS first_author_country
    FROM works w
    JOIN authorships a ON w.openalex_id = a.openalex_id AND a.position = 'first'
    WHERE w.method_type IS NOT NULL
      AND a.institution_country IS NOT NULL
""").fetchdf()

geo_methods['method_name'] = geo_methods['method_type'].map(method_map)

# Define "causal" methods
causal_methods = ['M01', 'M02']  # RCT, Quasi-Experimental
geo_methods['is_causal'] = geo_methods['method_type'].isin(causal_methods)

# Top 20 countries by paper count
top_countries = geo_methods['first_author_country'].value_counts().head(20).index

causal_by_country = (
    geo_methods[geo_methods['first_author_country'].isin(top_countries)]
    .groupby('first_author_country')
    .agg(
        total=('openalex_id', 'count'),
        causal=('is_causal', 'sum'),
    )
    .assign(causal_rate=lambda x: x['causal'] / x['total'])
    .sort_values('causal_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 8))
causal_by_country['causal_rate'].plot(kind='barh', ax=ax, color='#2171b5')
ax.set_xlabel('Proportion of papers using causal methods (RCT + quasi-experimental)')
ax.set_title('Causal Inference Method Usage by First-Author Country (Top 20)')
for i, (idx, row) in enumerate(causal_by_country.iterrows()):
    ax.text(row['causal_rate'] + 0.005, i,
            f"{row['causal_rate']:.0%} (n={row['total']})", va='center', fontsize=9)
fig.tight_layout()
plt.show()

---
## D4. Methods Diffusion Curves

When did each method start being widely used across global health topics? Diffusion curves show the spread of methodological innovation.

In [ ]:
# For each method, count how many distinct topic categories used it each year
diffusion = (
    works
    .groupby(['publication_year', 'method_type'])
    ['topic_category'].nunique()
    .reset_index(name='n_topics_using')
)
diffusion['method_name'] = diffusion['method_type'].map(method_map)

# Focus on methods of interest (newer/emerging methods)
interesting_methods = ['M02', 'M06', 'M07', 'M08', 'M09', 'M10', 'M11']
method_names = [method_map.get(m, m) for m in interesting_methods]

fig, ax = plt.subplots(figsize=(14, 6))
for method_id in interesting_methods:
    md = diffusion[diffusion['method_type'] == method_id]
    if len(md) > 0:
        ax.plot(md['publication_year'], md['n_topics_using'], 'o-',
                label=method_map.get(method_id, method_id), markersize=5)

ax.set_xlabel('Year')
ax.set_ylabel('Number of topic categories using this method')
ax.set_title('Methods Diffusion Curves — Spread of Methods Across Topics')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_ylim(0, None)
fig.tight_layout()
plt.show()

# Also show method share over time (volume, not just breadth)
method_year_counts = (
    works
    .groupby(['publication_year', 'method_name'])
    ['openalex_id'].count()
    .reset_index(name='n_papers')
)
year_totals = method_year_counts.groupby('publication_year')['n_papers'].sum()
method_year_counts['share'] = method_year_counts.apply(
    lambda r: r['n_papers'] / year_totals[r['publication_year']], axis=1
)

fig, ax = plt.subplots(figsize=(14, 6))
for method in interesting_methods:
    mname = method_map.get(method, method)
    md = method_year_counts[method_year_counts['method_name'] == mname]
    if len(md) > 0:
        ax.plot(md['publication_year'], md['share'], 'o-', label=mname, markersize=5)

ax.set_xlabel('Year')
ax.set_ylabel('Share of all papers')
ax.set_title('Method Usage Share Over Time')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
fig.tight_layout()
plt.show()

---
## D5. Manual Review of Top 10 Gap Cells

Not every gap is a genuine research opportunity — some topic-method combinations don't make sense (e.g., RCTs for epidemiology). Review each gap and classify as:
- **Genuine opportunity** — this method should be applied to this topic but isn't
- **Appropriate absence** — this method doesn't fit this topic
- **Emerging** — early signs of growth

In [ ]:
# Display top 10 gaps for manual review
review_gaps = meaningful_gaps.head(10).copy()
review_gaps['assessment'] = ''  # Fill in manually
review_gaps['notes'] = ''       # Fill in manually

print('Review each gap below and add your assessment:')
print('  - "opportunity" = genuine research gap worth filling')
print('  - "appropriate" = method doesn\'t apply to this topic')
print('  - "emerging" = early signs of adoption')
print()

for i, (_, row) in enumerate(review_gaps.iterrows()):
    print(f'{i+1}. {row["topic"]} × {row["method"]}')
    print(f'   Observed: {row["observed"]:.0f} papers, Expected: {row["expected"]:.1f}, Z: {row["z_score"]:.1f}')
    print()

# After manual review, update the assessment column:
# review_gaps.loc[review_gaps.index[0], 'assessment'] = 'opportunity'
# review_gaps.loc[review_gaps.index[0], 'notes'] = 'More RCTs needed in this area'
# etc.

---
## Save Results to DuckDB

In [ ]:
con.close()

con_write = duckdb.connect(DB)

# Topic × method matrix (z-scores)
z_flat = gaps_df.copy()
con_write.execute('CREATE OR REPLACE TABLE methods_gap_matrix AS SELECT * FROM ?', [z_flat])

# Topic × method observed counts (for the dashboard)
topic_method_counts = (
    works
    .groupby(['topic_category', 'topic_name', 'method_type', 'method_name'])
    ['openalex_id'].count()
    .reset_index(name='n_papers')
)
con_write.execute('CREATE OR REPLACE TABLE topic_method_matrix AS SELECT * FROM ?', [topic_method_counts])

con_write.close()
print('Lens D results saved to DuckDB.')